In [1]:
import sys 
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

from anndata import AnnData
import mudata as md
from scvi.model import TOTALVI
import anndata as ad
import scipy
import resource
import numpy as np
import pandas as pd
import scipy.io as sio
import scanpy as sc
from copy import deepcopy
import timeit

from sklearn.cluster import KMeans
import h5py
from scipy.sparse import csr_matrix

#==== method specific ==== 
import scvi

scvi.settings.seed = 420

/data1/chengyue/miniconda3/envs/multivi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data1/chengyue/miniconda3/envs/multivi/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/data1/chengyue/miniconda3/envs/multivi/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
Seed set to 420


In [2]:
def read_mtx_folder(dir_path,mod_key,var_list,obs_list):
    mtx_path = []
    tsv_path = []
    for file in os.listdir(dir_path):
        if file.endswith(".mtx"):
            mtx_path.append(os.path.join(dir_path, file))
    if(len(mtx_path)==0):
        print("no .mtx file found, exiting function")
        return
    counts = sio.mmread(mtx_path[0])
    #if type(counts) != "scipy.sparse.csr.csr_matrix": counts = counts.tocsr()
    if not isinstance(counts, scipy.sparse.csr_matrix): 
        counts = counts.tocsr()
    
    var_v = []
    for i in var_list:
        var_v.append(pd.read_csv(os.path.join(dir_path,i+".tsv"), sep = '\t'))#csv
    var_df = pd.concat(var_v, axis=1)
    #var_df = var_df.set_axis(var_list, axis='columns')
    
    obs_v = []
    for i in obs_list:
        obs_v.append(pd.read_csv(os.path.join(dir_path,i+".csv"), sep = '\t'))#csv
    obs_df = pd.concat(obs_v, axis=1)
    obs_df = obs_df.set_axis(obs_list, axis='columns')
    if(counts.shape[0]!=obs_df.shape[0]): counts=deepcopy(counts.transpose())
    adata = AnnData(counts,obs=obs_df,var=var_df)
    adata.obs = adata.obs.set_axis(list(adata.obs["barcodes"]),axis="index")
    adata.var= adata.var.set_axis(list(adata.var['feature_name']),axis="index")
    adata.var["modality"] = mod_key
    return(adata)

In [3]:
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, adjusted_mutual_info_score
from sklearn.metrics import fowlkes_mallows_score, homogeneity_score, completeness_score, v_measure_score, silhouette_score
def eva(y_true, y_pred):
    nmi = nmi_score(y_true, y_pred)
    ari = ari_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    fmi = fowlkes_mallows_score(y_true, y_pred)
    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v = v_measure_score(y_true, y_pred)
    return nmi, ari, ami, fmi, hom, com, v

def eva_nolabel(data, y):
    db = davies_bouldin_score(data, y)
    ch = calinski_harabasz_score(data, y)
    asw = silhouette_score(data, y)
    return db, ch, asw

In [ ]:
def run_multivi_fn(in_dir, out_dir,label_file=None):
    # save latent representation and model 
    start = timeit.default_timer()
    scvi.settings.seed = 420
    adata_prna = read_mtx_folder(os.path.join(in_dir,"paired_RNA/"),
                                       "Gene Expression",
                                       ["gene"],
                                       ["barcodes"])
    
    adata_patac = read_mtx_folder(os.path.join(in_dir,"paired_ATAC/"),
                                       "Peaks",
                                       ["peak"],
                                       ["barcodes"])

    cell_name = pd.read_csv('./human_brain_3k/label.csv', usecols=[1])
    cell_type, y = np.unique(cell_name, return_inverse=True)
 
    adata_prna.var_names_make_unique()
    
    
    adata_prna.var_names = ["rna_" + str(gene) for gene in adata_prna.var_names]
    
    adata_patac.var_names = ["atac_" + str(gene) for gene in adata_patac.var_names]
    
    adata_patac.obs['modality']='Peaks'
    adata_patac.var['modality']='Peaks'
    
    adata_prna.obs['modality']='Gene Expression'
    adata_prna.var['modality']='Gene Expression'

    # Horizontally stack two modalities of paired dataset 
    adata_paired = AnnData(scipy.sparse.hstack((deepcopy(adata_prna.X), 
                                            deepcopy(adata_patac.X)), 
                                           format='csr'),
                           obs = deepcopy(adata_prna.obs),
                           var = pd.concat([deepcopy(adata_prna.var[["modality"]]),deepcopy(adata_patac.var[["modality"]])]))
    # organize_mulitome_anndatats: concatenate paired and two unpaired anndata
    adata_mvi = scvi.data.organize_multiome_anndatas(adata_paired)#, adata_urna, adata_uatac)
    # # gene features need to be before chromatin peaks (algorithm assumption)
    adata_mvi = adata_mvi[:, adata_mvi.var["modality"].argsort()].copy()
    sc.pp.filter_genes(adata_mvi, min_cells=int(adata_mvi.shape[0] * 0.01))
    # setup batch annotation
    scvi.model.MULTIVI.setup_anndata(adata_mvi, batch_key='modality')
    # setup model 
    mvi = scvi.model.MULTIVI(
        adata_mvi,
        n_genes=(adata_mvi.var['modality']=='Gene Expression').sum(),
        n_regions=(adata_mvi.var['modality']=='Peaks').sum(), # Peaks
    )
    # train 
    mvi.train()
    os.makedirs(out_dir, exist_ok=True)
    # get latent representation 
    adata_mvi.obsm["MultiVI_latent"] = mvi.get_latent_representation()
    print(adata_mvi.obsm["MultiVI_latent"])
    adata_mvi.obs = adata_mvi.obs.set_axis([s. split("_", 1)[0] for s in adata_mvi.obs.index], axis='index')

    # extract latent representation
    res_df = pd.DataFrame(adata_mvi.obsm['MultiVI_latent'],index=adata_mvi.obs.index)
    print(res_df)
    
    n_clusters = int(max(y) - min(y) + 1)   
    
    kmeans = KMeans(n_clusters = n_clusters, n_init=20)
    y_pred_z = kmeans.fit_predict(res_df)   
    csv_out = os.path.join(out_dir, "multivi_result.csv")
    res_df.to_csv(csv_out)
    pred_out = os.path.join(out_dir, "multivi_pred.csv")
    np.savetxt(pred_out, y_pred_z)
    y = y.ravel()
    print('y:', y.shape)
    print('y_pred_z:', y_pred_z.shape)
    nmi, ari, ami, fmi, hom, com, v = eva(y, y_pred_z)
    db, ch, asw = eva_nolabel(res_df, y_pred_z)
    print('nmi, ari, ami, fmi, hom, com, v:', nmi, ari, ami, fmi, hom, com, v)
    print('db, ch, asw:', db, ch, asw)
    
    
    # save latent representation and model 
    os.makedirs(os.path.join(out_dir,"multivi"), exist_ok=True)
    os.makedirs(os.path.join(out_dir,"runtime"), exist_ok=True)
    
    csv_out = os.path.join(out_dir, "multivi","multivi_result.csv")
    res_df.to_csv(csv_out)
    pred_out = os.path.join(out_dir, "multivi","multivi_pred.csv")
    np.savetxt(pred_out, y_pred_z)
    model_out = os.path.join(out_dir,"multivi","trained_multivi")
    mvi.save(model_out, overwrite=True)
    stop = timeit.default_timer()
    print('Time(s): ', stop - start)  
    # record time 
    runtime_out = os.path.join(out_dir,"runtime","multivi_runtime.txt")
    print(stop - start,  file=open(runtime_out, 'w'))
    print("------ Done ------")

    stop = timeit.default_timer()
    max_mem = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(f"max_mem: , {max_mem/1024/1024:.2f}GB")
    print('Time(s): ', stop - start)  
    

In [ ]:
mvi = run_multivi_fn('./human_brain_3k/','./human_brain_3k/multivi-output/')

Seed set to 420
/home/chengyue/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/chengyue/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/chengyue/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/chengyue/.local/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/tmp/ipykernel_2187559/1566592328.py:64: DeprecationWarning: MULTIVI is supposed to work with MuData. the use of anndata is depr

Epoch 355/500:  71%|███████   | 355/500 [12:53<05:15,  2.18s/it, v_num=1, train_loss_step=3.42e+4, train_loss_epoch=3.49e+4]
Monitored metric reconstruction_loss_validation did not improve in the last 50 records. Best score: 37888.402. Signaling Trainer to stop.
[[ 0.48522857 -0.9192352   0.51082665 ...  0.06250063  0.71728516
   0.75705886]
 [ 0.9886221  -0.5952169   0.58115935 ...  0.7602341   0.5621666
   1.0835719 ]
 [-0.6259856  -0.01122661  0.49256143 ...  0.8321754   1.1670606
   0.86260664]
 ...
 [ 0.27535152  1.2867035   0.3682585  ...  0.94887656 -0.7475858
   0.00223654]
 [-0.670828    1.1833775   0.6086093  ...  1.6293919   0.58898413
   0.1724118 ]
 [-0.25321084 -0.6477779   0.01227512 ... -2.205065   -1.6402569
   0.11602874]]
                          0         1         2         3         4   \
AAACAGCCATAGACTT-1  0.485229 -0.919235  0.510827 -2.162174  0.371279   
AAACAGCCATTATGCG-1  0.988622 -0.595217  0.581159 -1.712468  0.330002   
AAACCAACATAGACCC-1 -0.625986 -0.0

/tmp/ipykernel_2187559/1566592328.py:90: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  n_clusters = int(max(y) - min(y) + 1)


y: (3233,)
y_pred_z: (3233,)
nmi, ari, ami, fmi, hom, com, v: 0.6540698665928608 0.49005735774488873 0.6520872845914171 0.5491299746708462 0.6564347263480372 0.6517219848740391 0.6540698665928609
db, ch, asw: 1.0928386658946236 1105.6364 0.33403292
Time(s):  784.8752237521112
------ Done ------
max_mem: , 6.74GB
Time(s):  784.8756850734353
